# Seller Listing Factory: Multi-Agent E-Retail with Google ADK

**What you will learn:**

- **Agent**: A single AI worker that performs one specific task using instructions
- **instruction**: The text that tells an agent what to do
- **output_key**: A name each agent uses to save its result so the next agent can read it
- **Workflow**: An assembly line that runs multiple agents in sequence
- **{placeholders}**: Curly braces in instructions that read saved results from previous agents

**The pipeline:**

```
Raw notes → Extractor → Copywriter → SEO → Compliance → Final listing
```

We will build this step by step: first one agent, then two chained together, then add more stages.

## Setup

Install Google ADK (the framework for building agents) and LiteLlm (a translator that lets ADK talk to Anthropic's Claude instead of only Google's Gemini).

In [ ]:
# Install the two packages we need
#!pip install -q google-adk==2.10.0 litellm==1.102.1

Check that the Anthropic API key is available. LiteLlm reads `ANTHROPIC_API_KEY` from the environment automatically.

In [ ]:
import os

# Check if the API key exists in the environment
if os.environ.get("ANTHROPIC_API_KEY"):
    print("✓ API key found")
else:
    print("✗ API key missing - set ANTHROPIC_API_KEY")

Define the model once so we can reuse it for all agents. We use Claude Haiku 4.5 (the cheapest Anthropic model) through LiteLlm.

In [ ]:
from google.adk.models import LiteLlm

# Define the model once and reuse it for all agents
# LiteLlm translates between ADK (built for Gemini) and Anthropic
MODEL = LiteLlm(model="anthropic/claude-haiku-4-5-20251001")

## Step 1: Your First Agent

An **agent** is an AI worker that follows instructions. We create an "Attribute Extractor" that turns messy seller notes into clean attributes (material, sizes, price, etc.).

We also define a helper function `ask()` to run an agent and print its reply.

In [ ]:
from google.adk import Agent, Runner
from google.adk.sessions import InMemorySessionService

# Create the first agent: extracts attributes from messy notes
extractor = Agent(
    name="extractor",  # A name to identify this agent
    model=MODEL,  # The AI model to use
    instruction="Extract product attributes: material, sizes, price, origin, care instructions. List them clearly.",
    output_key="attributes"  # Save the result under this name for the next agent
)

# Helper function to run an agent and print the reply
async def ask(agent, text):
    # Runner executes the agent
    runner = Runner(app_name="demo", agent=agent, session_service=InMemorySessionService())
    # run_debug sends a message and returns events (the agent's replies)
    events = await runner.run_debug(text, quiet=True)
    # Print the text from each event
    for e in events:
        if e.content:
            print(e.content.parts[0].text)

### Understanding async/await

**What is `async` and `await`?**

Think of a restaurant: you place an order and the kitchen works while you wait, instead of you freezing at the counter until the food is ready.

In Python:
- `async` marks a function that can wait for things (like waiting for the AI to reply)
- `await` means "wait here for this to finish, but let other things run in the meantime"

In Jupyter, just write `await` in front of the call. That's all you need to remember.

In [ ]:
# Messy seller input
raw_notes = """
Hand block-printed cotton kurta from Jaipur. 100% organic cotton, best in India!! 
Sizes M L XL XXL available. Price ₹899. Guaranteed to last forever. 
Traditional Rajasthani print. Machine wash cold.
"""

# Run the extractor agent
await ask(extractor, raw_notes)

## Step 2: Chain Two Agents

Now we add a second agent (a "Copywriter") that writes a title and description from the extractor's output.

**Key concepts:**
- **output_key**: Each agent saves its result under a name (e.g. "attributes")
- **{placeholders}**: The next agent's instruction uses curly braces to read that saved result: `{attributes}`
- **Workflow**: An assembly line that runs agents in sequence using `edges` (a list of connections)

A **Workflow** is like a factory assembly line: the first agent does its job, saves the result, then the second agent picks it up and does the next step.

In [ ]:
from google.adk import Workflow

# Second agent: writes a title and description
copywriter = Agent(
    name="copywriter",
    model=MODEL,
    # {attributes} reads the output saved by the extractor
    instruction="Write a compelling product title and a 2-sentence description from these attributes: {attributes}",
    output_key="draft_listing"
)

# Create a two-stage pipeline
pipeline_step2 = Workflow(
    name="factory",
    # edges is a list of connections: (from, to)
    # "START" means the beginning of the pipeline
    edges=[
        ("START", extractor),  # First: run extractor
        (extractor, copywriter),  # Then: run copywriter
    ]
)

# Run the two-stage pipeline
await ask(pipeline_step2, raw_notes)

## Step 3: Add an SEO Agent

Adding a new stage to the assembly line is simple: just add another agent to the `edges` list.

This agent generates search keywords and a short meta title for SEO.

In [ ]:
# Third agent: generates SEO keywords and meta title
seo = Agent(
    name="seo",
    model=MODEL,
    instruction="From this listing: {draft_listing}, generate 5 search keywords and a short meta title (under 60 characters).",
    output_key="seo_data"
)

# Add the SEO agent to the pipeline
pipeline_step3 = Workflow(
    name="factory",
    edges=[
        ("START", extractor),
        (extractor, copywriter),
        (copywriter, seo),  # Add the SEO stage
    ]
)

await ask(pipeline_step3, raw_notes)

## Step 4: Add a Compliance Reviewer

Marketplaces ban exaggerated claims like "100% organic", "best in India", "guaranteed", "cures", "government approved".

This agent checks the draft listing against a list of banned phrases and rewrites it to remove them, noting what changed.

**Why this matters:** Real marketplaces reject listings with unverified claims. Automating compliance saves time and avoids rejection.

In [ ]:
# Fourth agent: removes banned claims
compliance = Agent(
    name="compliance",
    model=MODEL,
    instruction="""
Review this listing: {draft_listing}

Check for these BANNED claims:
- "100% organic" (unless certified)
- "best in India" or similar superlatives
- "guaranteed" or "forever"
- "cures" or medical claims
- "government approved"

Rewrite the listing without these claims. Note what you changed.
""",
    output_key="final_listing"
)

# Add compliance to the pipeline
pipeline_step4 = Workflow(
    name="factory",
    edges=[
        ("START", extractor),
        (extractor, copywriter),
        (copywriter, seo),
        (copywriter, compliance),  # Compliance reviews the draft
    ]
)

await ask(pipeline_step4, raw_notes)

## Step 5: Run the Full Factory

Let's run all four stages and see each step's output to watch the assembly line work.

In [ ]:
# Run the full pipeline with verbose output
runner = Runner(app_name="demo", agent=pipeline_step4, session_service=InMemorySessionService())
events = await runner.run_debug(raw_notes, quiet=False, verbose=True)

## Recap and Exercises

**What we built:**
1. **Agent**: Each AI worker has a name, model, instruction, and output_key
2. **Workflow**: An assembly line connecting agents with `edges`
3. **{placeholders}**: How agents read each other's saved results
4. **Pipeline**: Raw notes → Extractor → Copywriter → SEO → Compliance → Final listing

**Try these exercises:**

1. **Add a Translator**: Create a new agent that translates the final listing to Hindi. Add it after the compliance agent.

2. **Update banned claims**: Add "cheapest" and "number 1" to the compliance agent's banned list.

3. **Change the tone**: Modify the copywriter's instruction to write in a premium luxury tone OR a budget-friendly tone. See how the output changes.